# Optimization Campaign (HITL)

**Prerequisites:** TermNorm backend at `http://127.0.0.1:8000` | Groq API key in `.env` | Restart kernel after first sync

**Workflow:** Setup → Data → Explore → Optimize → Results

In [ ]:
%load_ext autoreload
%autoreload 2

from promptpotter.display.campaign import *

# --- Services ---
session = await init_services()
TASK_DESCRIPTION = load_task_description(
    r"C:\Users\dsacc\OfficeAddinApps\TermNorm-excel\backend-api\config\LCA_INPUT_PATTERNS.md"
)

# --- Campaign config ---
# ALL experiment knobs live here — no hidden defaults in service code.
campaign_config = {
    "eval_sample_size": 15,  # queries per eval step (0 = all)
    "scan_sample_size": 10,  # queries per scan/grid point (can be smaller)
    "exclude_nodes": ["llm_ranking"],  # nodes to skip (e.g. ["entity_profiling"])
    # --- Backend node overrides ---
    # Override values from GET /pipeline. Nested format: {"node": {"param": value}}.
    # See show_pipeline_snapshot() output for available params per node.
    "pipeline_overrides": {
        "web_search": {
            "max_sites": 20,
            "num_results": 20,
            "content_char_limit": 800,
        },
        "entity_profiling": {
            "model": "openai/gpt-oss-120b",
            "max_tokens": 4000,
        },
        "llm_ranking": {
            "model": "openai/gpt-oss-120b",
        },
    },
    "optimization": {
        # --- Core loop ---
        "l1_patience": 2,  # consecutive non-improvements before stop/escalate
        "max_rounds": None,  # None = unlimited
        "n_variants": 5,  # candidates per round
        "creativity": 0.7,  # temperature for candidate generation
        "improvement_threshold": 0.01,  # accuracy delta to count as improvement
        "seed": 42,  # subsampling seed (reproducibility)
        "max_failures": 15,  # failure examples fed to LLM candidate generation
        # --- Escalation ---
        "degradation_threshold": 0.4,  # fraction of degraded queries to trigger escalation
        "backend_warning_threshold": 2,  # degradation resets before backend advisory
        "enable_l2": True,  # L2 refine_context on escalation
        "enable_l3": True,  # L3 modify_plan on L2 stall
        "l2_patience": 2,  # L2 stalls before L3
        "l3_patience": 1,  # L3 stalls before stop
        "l2_temperature": 0.3,  # LLM temperature for L2 transitions
        "l3_temperature": 0.5,  # LLM temperature for L3 transitions
        # --- Critique ---
        "enable_critique": True,  # critique agent between generate/evaluate
    },
    "optimizer_llm": {
        # "model":       "moonshotai/kimi-k2-instruct-0905",  # 10x more expensive
        "model": "openai/gpt-oss-120b",
        "provider": "groq",
        "temperature": 0.4,
        # --- Anthropic (cost: opus >> sonnet >> haiku) ---
        # "model": "claude-opus-4-6",
        # "model": "claude-sonnet-4-6",
        # "model": "claude-haiku-4-5-20251001",
        "max_tokens": 2000,
    },
    "pipeline_params": None,  # set by configure_pipeline()
}

# --- Pipeline snapshot & params ---
pipeline_config_full = await show_pipeline_snapshot(session)
pipeline_params = configure_pipeline(session, campaign_config)

In [ ]:
# @title Load data & evaluation context
RUN_BASELINE = True  # Set False only during iterative dev

train_data, index_terms = prepare_datasets(
    session.store,
    session.backend_id,
    excel_path=r"C:\Users\dsacc\Desktop\project-TermNorm\OneDrive_2025-07-02\Austausch Beispiele\Prozessnamen\BOM-example.xlsx",
)
session.index_terms = index_terms

baseline_ps, dataset, campaign_rounds, baseline_results = await prepare_eval_context(
    session,
    train_data,
    campaign_config,
    run_baseline=RUN_BASELINE,
    pipeline_params=pipeline_params,
)

In [ ]:
# @title Experiment dashboard
EXPERIMENT_ID = None  # Set to hex ID to resume (e.g. '68e2c5')
pipeline_params = locals().get("pipeline_params")  # preserve on re-run

pipeline_params = show_experiment_dashboard(
    session=session,
    experiment_id=EXPERIMENT_ID,
    campaign_config=campaign_config,
    dataset=dataset,
    baseline_prompt_fields=campaign_rounds[0]["prompt_fields"].model_dump()
    if campaign_rounds
    else None,
    pipeline_params=pipeline_params,
)

## 3. Explore

Exploration via **Smart Search** (scan advisor + sensitivity scan).

In [ ]:
# @title Task context + scan advisor
ADVISOR_MODEL = "openai/gpt-oss-120b"  # model for scan advisor LLM call

task_context = await decompose_task_context(TASK_DESCRIPTION, campaign_config, session)

# preview_advisor_prompt(campaign_config, session, task_description=task_context, raw=True)
# advisory, scan_variants, schema_labels = await run_scan_advisor(
#     campaign_config, session,
#     task_description=task_context,
#     model=ADVISOR_MODEL,
# )

In [ ]:
# @title Scan variant config (edit suggested values or add your own)
# Schema axes: mutation tuples ("-", path), ("+", path, type, req, desc),
# ("~", old, new, type, req, desc). Non-schema axes: plain value lists.

scan_sample_size = 3  # queries per scan variant (0 = use all)

scan_variants = {
    # ── Token matching ───────────────────────────────────────────────────
    "token_matching": {
        "max_token_candidates": [10, 30, 50],
    },
    # ── Web search ────────────────────────────────────────────────────────
    "web_search": {
        "query_prefix": [
            # --- Database-oriented ---
            "ecoinvent",
            "GaBi",
            "ecoinvent LCA database material name",
            "material composition LCA",
            "what is",
            "technical data sheet",
            "product specification",
            "manufactured from",
            "production process for",
        ],
        "max_sites": [3, 7, 12],
        "num_results": [5, 20, 40],
        "content_char_limit": [400, 800, 1500],
    },
    # ── Entity profiling ──────────────────────────────────────────────────
    "entity_profiling": {
        "temperature": [0.0, 0.3, 0.7],
        "raw_content_limit": [1000, 2500, 8000],
        "profiling_schema": [
            # --- LCA-database-specific (original) ---
            [
                [
                    "+",
                    "geography_scope",
                    "array",
                    False,
                    "Relevant ecoinvent/SimaPro geography codes inferred from context, e.g. GLO, RER, CH, RNA",
                ],
                [
                    "+",
                    "database_format_hint",
                    "string",
                    False,
                    "Best-guess ecoinvent-style name fragment for this entity, e.g. 'market for polyethylene, high density'",
                ],
            ],
            [
                [
                    "+",
                    "lca_database_names",
                    "array",
                    True,
                    "Likely ecoinvent or GaBi database entry names that would match this entity, using standard LCA database naming conventions like 'market for X | X | cut-off, U'",
                ]
            ],
            [
                ["-", "manufacturing_processes"],
                ["-", "applications"],
                [
                    "+",
                    "lca_database_names",
                    "array",
                    True,
                    "Likely ecoinvent or GaBi database entry names for this entity using standard LCA naming conventions",
                ],
            ],
            [
                ["-", "applications"],
                [
                    "+",
                    "ecoinvent_candidate_names",
                    "array",
                    True,
                    "Exact ecoinvent activity names this entity most likely maps to",
                ],
            ],
            [
                ["-", "manufacturing_processes"],
                [
                    "+",
                    "database_search_tokens",
                    "array",
                    True,
                    "Optimized search tokens for LCA database lookup including spelling variants",
                ],
            ],
            # --- Minimalist: strip to core matching signals ---
            [
                ["-", "applications"],
                ["-", "manufacturing_processes"],
                ["-", "notes"],
                ["-", "technical_specifications"],
            ],
            # --- Chemical identity ---
            [
                [
                    "+",
                    "cas_number",
                    "string",
                    False,
                    "CAS registry number if identifiable from context",
                ],
                [
                    "+",
                    "chemical_formula",
                    "string",
                    False,
                    "Chemical formula or molecular structure notation",
                ],
            ],
            # --- Trade name decoding ---
            [
                [
                    "~",
                    "key_properties",
                    "trade_names",
                    "array",
                    True,
                    "Known commercial/trade names and brand names for this material, e.g. Makrolon=polycarbonate, Delrin=POM",
                ]
            ],
            # --- Material hierarchy (specific→generic) ---
            [
                [
                    "+",
                    "material_hierarchy",
                    "array",
                    False,
                    "Classification chain from specific to generic, e.g. [Makrolon 2805, polycarbonate, thermoplastic, polymer]",
                ]
            ],
            # --- Process-centric (flip perspective from material to process) ---
            [
                [
                    "~",
                    "applications",
                    "production_route",
                    "string",
                    False,
                    "Primary production/manufacturing route e.g. injection molding, extrusion, casting",
                ],
                [
                    "~",
                    "notes",
                    "form_factor",
                    "string",
                    False,
                    "Physical form: granulate, sheet, rod, wire, powder, liquid, film",
                ],
            ],
            # --- Standards-focused ---
            [
                [
                    "+",
                    "applicable_standards",
                    "array",
                    False,
                    "DIN/ISO/EN/ASTM standards that reference or define this material",
                ],
                ["-", "applications"],
            ],
            # --- Werkstoff / alloy code decoding ---
            [
                [
                    "+",
                    "material_code_decoded",
                    "string",
                    False,
                    "Decoded meaning of any material code, Werkstoff number, or alloy designation present in the input",
                ],
                [
                    "+",
                    "base_material",
                    "string",
                    False,
                    "The fundamental base material, e.g. brass, steel, polycarbonate",
                ],
            ],
            # --- Functional equivalence ---
            [
                [
                    "~",
                    "applications",
                    "functional_unit",
                    "string",
                    False,
                    "The functional unit this material serves, e.g. structural plastic, electrical insulation, food-grade packaging",
                ],
                [
                    "+",
                    "substitutes",
                    "array",
                    False,
                    "Materials that could serve the same functional role",
                ],
            ],
        ],
    },
    # ── Prompt fields ────────────────────────────────────────────────────
    "thinking_style": [
        "Think step-by-step: isolate distinguishing features → compare each candidate → assign scores.",
        "Consider the most likely interpretation first, then check alternatives.",
        "Reason by elimination: discard obviously wrong candidates, then rank the rest.",
    ],
}
scan_variants, schema_labels = resolve_scan_variants(scan_variants, session=session)

In [ ]:
# @title Run sensitivity scan
scan_baseline_sp, scan_df, axis_profiles = await run_sensitivity_scan(
    baseline_ps,
    campaign_config,
    scan_variants,
    dataset,
    scan_sample_size=scan_sample_size,
    session=session,
    experiment_id=EXPERIMENT_ID or "",
)

In [ ]:
# @title Scan analytics
difficulty_df = show_scan_analytics(scan_df, axis_profiles, session)

In [ ]:
# @title Select scan winner & seed campaign
best_sp = seed_campaign_from_scan(
    scan_df,
    axis_profiles,
    scan_baseline_sp,
    scan_variants,
    campaign_rounds,
    campaign_config,
    pipeline_schema=session.pipeline_schema,
)

## 4. Optimize

Two modes: **Semi-automatic** (feedback cycle with patience-based auto-stop) or **Manual** (one round at a time).

In [ ]:
# @title Feedback cycle preflight
scan_context = show_feedback_preflight(
    campaign_rounds,
    dataset,
    campaign_config,
    pipeline_params=pipeline_params,
    scan_df=scan_df,
    axis_profiles=axis_profiles,
    scan_variants=scan_variants,
    difficulty_df=locals().get("difficulty_df"),
)

In [ ]:
# @title Run optimization (feedback cycle)
dev_reload()

campaign_rounds, _cycle_result = await run_optimization_notebook(
    campaign_rounds,
    dataset,
    campaign_config,
    session=session,
    pipeline_params=pipeline_params,
    scan_context=locals().get("scan_context"),
    experiment_id=EXPERIMENT_ID,
    task_context=task_context,
)

In [ ]:
# @title 5. Results — summary, save, sync
show_campaign_summary(campaign_rounds)
show_flip_tracking(campaign_rounds)
show_lineage_chain(campaign_rounds)

# --- Persist (T2: below the fold) ---
save_campaign_winner(
    campaign_rounds,
    campaign_config,
    session.store,
    session.backend_id,
    experiment_id=EXPERIMENT_ID,
)
sync_langfuse(
    session.store,
    session.backend_id,
    dataset_name="termnorm_ground_truth",
)